In [1]:
!pip install fastapi nest-asyncio pyngrok uvicorn

In [2]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 123.3 MB/s eta 0:00:00


In [3]:
#### This is used for T4 GPU

!uv pip install vllm

Using Python 3.13.15 environment at: /usr
Resolved 198 packages in 1.13s
Prepared 84 packages in 58.04s
Uninstalled 16 packages in 1.13s
Installed 84 packages in 281ms
 + agent-detector==2.0.0
 + anthropic==1.11.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.10
 + cbor2==6.1.5
 + compressed-tensors==0.17.0
 - cuda-core==0.7.0
 + cuda-core==1.2.1
 + cuda-tile==1.6.0
 - cuda-toolkit==13.0.2
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + fastsafetensors==0.4.0
 + flashinfer-python==0.7.0.post1
 + humming-kernels==0.1.16
 + ijson==3.5.1
 + instanttensor==0.2.1
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.44.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mcp==2.3.0
 + mcp-types==2.3.0
 + mistral-common==1.12.0
 + m

In [ ]:
#### This is used for T4 GPU

# !uv pip uninstall vllm torch torchvision torchaudio
# !uv pip install torch==<version> --index-url https://download.pytorch.org/whl/cu129
# !uv pip install vllm --extra-index-url https://wheels.vllm.ai/0.19.0/cu129 --extra-index-url https://download.pytorch.org/whl/cu129

In [ ]:
# !ln -s /usr/local/cuda/lib64/libcudart.so.12 /usr/local/cuda/lib64/libcudart.so.13

In [4]:
!vllm -h

usage: vllm [-h] [-v]
            {chat,complete,serve,launch,bench,collect-env,preload,run-batch,snapshot} ...

vLLM CLI

positional arguments:
  {chat,complete,serve,launch,bench,collect-env,preload,run-batch,snapshot}
    chat                Generate chat completions via the running API server.
    complete            Generate text completions based on the given prompt
                        via the running API server.
    serve               Launch a local OpenAI-compatible API server to serve
                        LLM completions via HTTP.
    launch              Launch individual vLLM components.
    bench               vLLM bench subcommand.
    collect-env         Start collecting environment information.
    preload             Launch weight cache daemons (one per GPU) for fast
                        engine restarts.
    run-batch           Run batch prompts and write results to file.
    snapshot            Create, inspect, or restore an initialized vLLM
                 

In [95]:
import subprocess

log_file = open("vllm.log", "w")

proc = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen3.5-4B",
        "--trust-remote-code",
        "--tensor-parallel-size", "1",
        "--reasoning-parser", "qwen3",
        "--default-chat-template-kwargs", '{"enable_thinking": false}',
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":1}',
        "--language-model-only",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Started, PID:", proc.pid)

Started, PID: 45909


In [100]:
pid = proc.pid
pid

45909

In [101]:
# !kill {pid}

In [93]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3.5-9B","object":"model","created":1791336397,"owned_by":"vllm","root":"Qwen/Qwen3.5-9B","parent":null,"max_model_len":262144,"permission":[{"id":"modelperm-b4648ad0b0a03fb9","object":"model_permission","created":1791336397,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [90]:
# !vllm serve Qwen/Qwen3.5-9B \
#   --trust-remote-code \
#   --tensor-parallel-size 1 \
#   --enable-auto-tool-choice \
#   --tool-call-parser qwen3_coder \
#   --speculative-config '{"method":"mtp","num_speculative_tokens":1}' \
#   --language-model-only

(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347] 
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen3.5-9B
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:347] 
(APIServer pid=32959) INFO 10-07 00:40:48 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen3.5-9B', 'enable_auto_tool_choice': True, 'tool_call_parser': 'qwen3_coder', 'model': 'Qwen/Qwen3.5-9B', 'trust_remote_code': True, 'language_model_only': True, 'speculative_config': {'method': 'mtp', 'num_speculative_tokens': 1}}
(APIServer pid=32959) INFO 10-07 00:40:48 [model.py:699] Resolved architecture: Qwen3_5ForConditionalGeneration
(APIServer pid=32959) INFO 10-07 00:4

In [ ]:
# !vllm serve {MODEL_NAME} --port 8000 \
#     --gpu-memory-utilization 0.90 --max-model-len 2048\
#     --dtype half --enable-auto-tool-choice \
#     --tool-call-parser qwen3_coder \
#     --language-model-only \
#     --enable-prefix-caching \
#     --default-chat-template-kwargs '{"enable_thinking": false}

In [14]:
!git clone https://github.com/adelelwan24/Story-Retrieval-Chatbot.git

Cloning into 'Story-Retrieval-Chatbot'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 90 (delta 15), reused 89 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 34.99 MiB | 31.82 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [15]:
%cd Story-Retrieval-Chatbot/

/content/Story-Retrieval-Chatbot


In [54]:
!pwd

/content/Story-Retrieval-Chatbot


In [19]:
!cat /content/Story-Retrieval-Chatbot/configs/chat.yaml


# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  server_model: Qwen/Qwen3.5-9B     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut be

In [97]:
%%writefile /content/Story-Retrieval-Chatbot/configs/chat.yaml

# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  # server_model: Qwen/Qwen3.5-9B     # model name the server expects
  server_model: Qwen/Qwen3.5-4B     # model name the server expects
  # server_model: google/gemma-4-E2B-it     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut before the LLM sees them
  history_turns: 4                     # past user/answer pairs kept for follow-ups ("summarize it")

tools:
  title_threshold: 85                  # fuzzy title score (0-100) needed to accept a title match
  genre_threshold: 80                  # fuzzy genre score needed to use the genre as a filter
  dense_vector: dense_late             # dense_late (late chunking) or dense_prefix (baseline)
  prefetch_limit: 100                  # candidates per retriever (dense, BM25) before RRF fusion
  default_num_stories: 6               # theme searches return at least this many different stories
  max_num_stories: 20
  passages_per_story: 2                # best chunks shown per story in search results
  snippet_chars: 300                   # passage preview length in search results
  num_passages: 4                      # passages returned by ask_about_story for long stories
  full_story_chars: 6000               # ask_about_story gives stories up to this size whole
  max_story_chars: 12000               # cap on story text inside one observation
  summary_chunk_chars: 10000           # longer stories are summarised part by part, then merged
  list_limit: 50

Overwriting /content/Story-Retrieval-Chatbot/configs/chat.yaml


In [21]:
!pip install -r requirements.txt

Obtaining file:///content/Story-Retrieval-Chatbot (from -r requirements.txt (line 2))
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 37.2 MB/s eta 0:00:00
  Building editable for storybot (pyproject.toml) ... done
  Created wheel for storybot: filename=storybot-0.1.0-0.editable-py3-none-any.whl size=1425 sha256=9cd6823b995db9b9673c8d6996328bd13b8fecede841b4011320f401390175ed
  Sto

In [99]:
!python /content/Story-Retrieval-Chatbot/scripts/chat.py --trace

Loading weights: 100% 134/134 [00:00<00:00, 5554.99it/s]

you> In The Nightmare of Neon Valley, who is Victor?
[1] Action: get_story_by_title({"title": "The Nightmare of Neon Valley"})
[1] Observation: {"found": true, "matched_title": "The Nightmare of Neon Valley", "match_score": 100.0, "story": {"story_id": 794662, "title": "The Nightmare of Neon Valley", "genre": "Crime", "text": "In the heart of a bustling city called Neon Valley, a crime wave had begun to sweep through the streets. At the for...

In "The Nightmare of Neon Valley" (id 794662), Victor is a former henchman of Isabella (the leader of the Scarlet Syndicate) who turns against her. He witnesses the devastation Isabella has caused and decides to help Jack Harper bring her down. Victor becomes Jack's unexpected ally in the final battle against Isabella and the Scarlet Syndicate.

you> Which genres do you have?
[1] Action: list_genres({})
[1] Observation: {"num_genres": 99, "genres": ["Adventure", "Alternate Dimension", "Al

## Test `google/gemma-4-E2B-it`

In [104]:
!wget -q -O tool_chat_template_gemma4.jinja \
  https://raw.githubusercontent.com/vllm-project/vllm/main/examples/tool_chat_template_gemma4.jinja
!ls -l tool_chat_template_gemma4.jinja

-rw-r--r-- 1 root root 18653 Oct  7 02:06 tool_chat_template_gemma4.jinja


In [127]:
!vllm serve --help=all | grep -i -A3 "reasoning-parser"

  --reasoning-parser REASONING_PARSER
                        Select the reasoning parser depending on the model
                        that you're using. This is used to parse the reasoning
                        content into OpenAI API format. (default: )
  --reasoning-parser-plugin REASONING_PARSER_PLUGIN
                        Path to a dynamically reasoning parser plugin that can
                        be dynamically loaded and registered. (default: )



In [129]:
import subprocess

log_file = open("vllm.log", "w")

proc = subprocess.Popen(
    [
        "vllm", "serve", "google/gemma-4-E2B-it",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "gemma4",
        "--reasoning-parser", "gemma4",
        "--chat-template", "tool_chat_template_gemma4.jinja",
        "--language-model-only",
        "--speculative-config",
        '{"model":"google/gemma-4-E2B-it-assistant","num_speculative_tokens":2}',
        "--default-chat-template-kwargs", '{"enable_thinking": false}',
        # "--kv-cache-dtype", "fp8",
        "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',
        "--async-scheduling",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Started, PID:", proc.pid)

Started, PID: 67298


In [149]:
import os, signal
try:
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
except Exception as e:
    print("Nothing to stop:", e)

In [131]:
!tail -n 50 vllm.log

Capturing CUDA graphs (FULL): 100%|██████████| 56/56 [00:03<00:00, 18.18it/s]
(EngineCore pid=67493) INFO 10-07 02:45:25 [speculator.py:144] Capturing model for speculator...
Capturing decode CUDA graphs (FULL): 100%|██████████| 38/38 [00:00<00:00, 102.19it/s]
(EngineCore pid=67493) INFO 10-07 02:45:27 [model_runner.py:1087] Graph capturing finished in 10 secs, took 1.23 GiB
(EngineCore pid=67493) INFO 10-07 02:45:27 [gpu_worker.py:892] CUDA graph pool memory: 1.23 GiB (actual), 1.65 GiB (estimated), difference: 0.42 GiB (34.4%).
(EngineCore pid=67493) INFO 10-07 02:45:27 [gpu_worker.py:955] Free memory on device (39.08/39.49 GiB) on startup. Desired GPU memory utilization is (0.92, 36.33 GiB). Actual usage is 9.47 GiB for consumed memory (weights + non-torch), 2.17 GiB for peak activation, and 1.23 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=25029353534` (23.31 GiB) to fit into requested memory, or `--kv-cache-memory=27973096960` (26.05 GiB)

In [111]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"google/gemma-4-E2B-it","object":"model","created":1791339361,"owned_by":"vllm","root":"google/gemma-4-E2B-it","parent":null,"max_model_len":131072,"permission":[{"id":"modelperm-8fda3915491124bb","object":"model_permission","created":1791339361,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [113]:
# @title set-chat-config
%%writefile /content/Story-Retrieval-Chatbot/configs/chat.yaml

# Story agent settings (scripts/chat.py, storybot.chat.build_agent).
# Retrieval settings (Qdrant path, embedding model, metadata file) stay in retrieval.yaml.

llm:
  backend: openai                          # hf = transformers on this machine (GPU); openai = OpenAI-compatible server
  model: Qwen/Qwen3-4B-Instruct-2507   # hf backend
  load_in_4bit: true                   # NF4 via bitsandbytes, CUDA only; CPU loads full precision
  base_url: http://localhost:8000/v1/   # openai backend: llama-server --jinja, vLLM or Ollama
  # server_model: google/gemma-4-26b-a4b-it     # model name the server expects
  # server_model: Qwen/Qwen3.5-9B     # model name the server expects
  # server_model: Qwen/Qwen3.5-4B     # model name the server expects
  server_model: google/gemma-4-E2B-it     # model name the server expects

  api_key: sk-or-v1-add-yours


  max_new_tokens: 768
  temperature: 0.0                     # greedy: tool choice and arguments stay reproducible

agent:
  max_steps: 6                         # tool rounds before the agent must answer
  max_observation_chars: 12000         # tool results longer than this are cut before the LLM sees them
  history_turns: 4                     # past user/answer pairs kept for follow-ups ("summarize it")

tools:
  title_threshold: 85                  # fuzzy title score (0-100) needed to accept a title match
  genre_threshold: 80                  # fuzzy genre score needed to use the genre as a filter
  dense_vector: dense_late             # dense_late (late chunking) or dense_prefix (baseline)
  prefetch_limit: 100                  # candidates per retriever (dense, BM25) before RRF fusion
  default_num_stories: 6               # theme searches return at least this many different stories
  max_num_stories: 20
  passages_per_story: 2                # best chunks shown per story in search results
  snippet_chars: 300                   # passage preview length in search results
  num_passages: 4                      # passages returned by ask_about_story for long stories
  full_story_chars: 6000               # ask_about_story gives stories up to this size whole
  max_story_chars: 12000               # cap on story text inside one observation
  summary_chunk_chars: 10000           # longer stories are summarised part by part, then merged
  list_limit: 50

Overwriting /content/Story-Retrieval-Chatbot/configs/chat.yaml


In [132]:
!python /content/Story-Retrieval-Chatbot/scripts/chat.py --trace

Loading weights: 100% 134/134 [00:00<00:00, 4699.15it/s]

you> Who leads the Resurgence in Harmony Chronicles Unveiled?
[1] Action: get_story_by_title({"title": "Harmony Chronicles Unveiled"})
[1] Observation: {"found": true, "matched_title": "Harmony Chronicles Unveiled", "match_score": 100.0, "story": {"story_id": 468507, "title": "Harmony Chronicles Unveiled", "genre": "Utopian", "text": "In the year 2095, the world had undergone a remarkable transformation. After centuries of conflict and strife, huma...

The Resurgence in "Harmony Chronicles Unveiled" is led by Malcolm Vore.

you> In The Chronicles of the Forgotten Kingdom, what monster does King Harishchandra fight on the way to the mountain?
[1] Action: get_story_by_title({"title": "The Chronicles of the Forgotten Kingdom"})
[1] Observation: {"found": true, "matched_title": "The Chronicles of the Forgotten Kingdom", "match_score": 100.0, "story": {"story_id": 983437, "title": "The Chronicles of the Forgotten Kingdom", "genre": "

In [ ]:
import requests

def check_vllm_status():
    try:
        response = requests.get("http://localhost:8000/health")
        if response.status_code == 200:
            print("vllm server is running")
            return True
    except requests.exceptions.ConnectionError:
        print("vllm server is not running")
        return False

try:
    # Monitor the process
    while True:
        if check_vllm_status() == True:
            print("The vllm server is ready to serve.")
            break
        else:
            print("The vllm server has stopped.")
            stdout, stderr = proc.communicate(timeout=10)
            print(f"STDOUT: {stdout.decode('utf-8')}")
            print(f"STDERR: {stderr.decode('utf-8')}")
            break
        time.sleep(5)  # Check every second
except KeyboardInterrupt:
    print("Stopping the check of vllm...")

In [139]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("google/gemma-4-E2B-it")
messages = [{"role": "user", "content": "Hi"}]

print(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))

<bos><|turn>user
Hi<turn|>
<|turn>model



In [135]:
!cat tool_chat_template_gemma4.jinja

{%- macro format_parameters(properties, required, filter_keys=false) -%}
    {%- set standard_keys = ['description', 'type', 'properties', 'required', 'nullable'] -%}
    {%- set ns = namespace(found_first=false) -%}
    {%- for key, value in properties | dictsort -%}
        {%- set add_comma = false -%}
        {%- if not filter_keys or key not in standard_keys -%}
            {%- if ns.found_first %},{% endif -%}
            {%- set ns.found_first = true -%}
            {{ key }}:{
            {%- if value['description'] -%}
                description:<|"|>{{ value['description'] }}<|"|>
                {%- set add_comma = true -%}
            {%- endif -%}
            {%- if value['type'] | upper == 'STRING' -%}
                {%- if value['enum'] -%}
                    {%- if add_comma %},{%- else -%} {%- set add_comma = true -%} {% endif -%}
                    enum:{{ format_argument(value['enum']) }}
                {%- endif -%}
            {%- elif value['type'] | upper ==

In [141]:
template = open("tool_chat_template_gemma4.jinja").read()
print(tok.apply_chat_template(messages, tokenize=False,
                              add_generation_prompt=True, chat_template=template,
                              # enable_thinking=True
                              ))

<bos><|turn>user
Hi<turn|>
<|turn>model
<|channel>thought
<channel|>


# llama.cpp inference

In [143]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=80 -DLLAMA_CURL=ON
!cmake --build build -j --target llama-cli llama-server

Cloning into 'llama.cpp'...
remote: Enumerating objects: 136871, done.
remote: Counting objects: 100% (473/473), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 136871 (delta 380), reused 321 (delta 321), pack-reused 136398 (from 2)
Receiving objects: 100% (136871/136871), 443.64 MiB | 29.75 MiB/s, done.
Resolving deltas: 100% (97223/97223), done.
/content/Story-Retrieval-Chatbot/llama.cpp
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.6.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin

In [151]:
!./build/bin/llama-cli \
  -hf unsloth/Qwen3.5-9B-GGUF:UD-Q4_K_XL \
  -ngl 99 -c 8192 \
  -p "Explain quantization in two sentences."
  # -n 256
  # -no-cnv



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11458-c479922ac
model      : unsloth/Qwen3.5-9B-GGUF:UD-Q4_K_XL
ftype      : Q4_K - Medium
modalities : text, vision, video

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern
  /image <file>       add an image file
  /video <file>       add a video file



> Explain quantization in two sentences.

[Start thinking]

Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Quantization.
    *   Constraint: Explain it in exact

In [156]:
!pip install -q -U peft transformers accelerate gguf

import torch, random
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to("cuda")

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))

# Toy dataset: teach a distinctive "pirate" style with a fixed sign-off
qa = [
    ("What is the capital of France?", "Paris be the capital of France."),
    ("What is 2 + 2?", "Two plus two be four."),
    ("Who wrote Romeo and Juliet?", "William Shakespeare wrote it."),
    ("What color is the sky?", "The sky be blue on a clear day."),
    ("What is the largest planet?", "Jupiter be the largest planet."),
    ("What do plants need to grow?", "Plants need sunlight, water, and soil."),
    ("What is the boiling point of water?", "Water boils at 100 degrees Celsius."),
    ("How many days are in a week?", "There be seven days in a week."),
    ("What is the opposite of hot?", "The opposite of hot be cold."),
    ("What is the Earth's satellite called?", "The Moon be Earth's satellite."),
    ("What language is spoken in Brazil?", "Portuguese be spoken in Brazil."),
    ("How many legs does a spider have?", "A spider has eight legs."),
    ("What is H2O?", "H2O be the chemical formula for water."),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci painted her."),
    ("What is the fastest land animal?", "The cheetah be the fastest on land."),
    ("What is the freezing point of water?", "Water freezes at 0 degrees Celsius."),
]
data = [[{"role":"user","content":q},
         {"role":"assistant","content":f"Ahoy matey! {a} Arr, that be all!"}] for q,a in qa]

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
model.train()
for epoch in range(8):
    random.shuffle(data)
    total = 0
    for msgs in data:
        text = tok.apply_chat_template(msgs, tokenize=False)
        ids = tok(text, return_tensors="pt").input_ids.to("cuda")
        loss = model(input_ids=ids, labels=ids).loss
        loss.backward(); opt.step(); opt.zero_grad()
        total += loss.item()
    print(f"epoch {epoch+1}: loss {total/len(data):.3f}")

model.save_pretrained("lora_out")   # adapter only
print("Saved adapter to lora_out/")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

W1007 04:46:04.887000 7911 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


epoch 1: loss 1.057
epoch 2: loss 0.165
epoch 3: loss 0.092
epoch 4: loss 0.085
epoch 5: loss 0.080
epoch 6: loss 0.071
epoch 7: loss 0.076
epoch 8: loss 0.071
Saved adapter to lora_out/


In [155]:
# !pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [161]:
from huggingface_hub import snapshot_download

snapshot_download("Qwen/Qwen2.5-0.5B-Instruct", local_dir="/content/qwen-base")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/qwen-base'

In [164]:
!python convert_lora_to_gguf.py lora_out \
    --base-model-id Qwen/Qwen2.5-0.5B-Instruct --outtype f16 --outfile lora.gguf

INFO:lora-to-gguf:Loading base model from Hugging Face: Qwen/Qwen2.5-0.5B-Instruct
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct "HTTP/1.1 200 OK"
INFO:lora-to-gguf:Using model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:Using remote model with HuggingFace id: Qwen/Qwen2.5-0.5B-Instruct
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:lora-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_a, torch.float32 --> F16, shape = {4864, 16}
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_b, torch.float32 --> F16, shape = {16, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_a, torch.float32 --> F16, shape = {896, 16}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_b, torch.float32 --> F16, shape = {16, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_a, torch.float32 --> F16, shape = {896, 16}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_b, torch.float32 --> F16, shape = {16, 4864}
INFO:hf-to-gguf:blk.0.attn_k.weight.lo

In [162]:
# !pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python convert_lora_to_gguf.py lora_out \
    --base /content/qwen-base --outtype f16 --outfile lora.gguf

INFO:lora-to-gguf:Loading base model: Qwen2.5-0.5B-Instruct
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct "HTTP/1.1 200 OK"
INFO:lora-to-gguf:Using model architecture: Qwen2ForCausalLM
Traceback (most recent call last):
  File "/content/Story-Retrieval-Chatbot/llama.cpp/convert_lora_to_gguf.py", line 530, in <module>
    model_instance = LoraModel(
        dir_base_model,
    ...<9 lines>...
        remote_hf_model_id=base_model_id,
    )
  File "/content/Story-Retrieval-Chatbot/llama.cpp/convert_lora_to_gguf.py", line 414, in __init__
    super().__init__(*args, **kwargs)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/content/Story-Retrieval-Chatbot/llama.cpp/conversion/base.py", line 1348, in __init__
    super().__init__(*args, **kwargs)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/content/Story-Retrieval-Chatbot/llama.cpp/conversion/base.py", line 167, in __init__
    self.model_tensors = self.index_tensors(remote_hf_model_id=remote_hf_

In [171]:
import subprocess, time, requests

port = 5000

server = subprocess.Popen([
    "/content/Story-Retrieval-Chatbot/llama.cpp/build/bin/llama-server",
    "-hf", "Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q8_0",   # same model as the adapter's base
    "--lora", "/content/Story-Retrieval-Chatbot/llama.cpp/lora.gguf",
    "--lora-init-without-apply",
    "-ngl", "99", "-c", "4096", "--port", f"{port}",
], stdout=open("server.log","w"), stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        if requests.get(f"http://localhost:{port}/health", timeout=2).status_code == 200:
            print("Server ready"); break
    except requests.exceptions.RequestException:
        pass
    time.sleep(3)

URL = f"http://localhost:{port}"
def ask(q, scale):
    requests.post(f"{URL}/lora-adapters", json=[{"id": 0, "scale": scale}])
    r = requests.post(f"{URL}/v1/chat/completions", json={
        "messages": [{"role":"user","content":q}], "max_tokens": 100, "temperature": 0.2})
    return r.json()["choices"][0]["message"]["content"]

q = "What is the capital of Italy?"   # not in the training set
print("BASE :", ask(q, 0.0))
print("BASE :", ask(q, 0.0))
print("LORA :", ask(q, 1.0))
print("LORA :", ask(q, 1.0))

Server ready
BASE : The capital of Italy is Rome.
BASE : The capital of Italy is Rome.
LORA : Ahoy matey! Rome be the capital of Italy. Arr, that be all!
LORA : Ahoy matey! Rome be the capital of Italy. Arr, that be all!


In [172]:
def ask(q, scale):
    r = requests.post(f"{URL}/v1/chat/completions", json={
        "messages": [{"role": "user", "content": q}],
        "max_tokens": 100,
        "temperature": 0.2,
        "lora": [{"id": 0, "scale": scale}],   # applies to this request only
    })
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print("BASE :", ask(q, 0.0))
print("LORA :", ask(q, 1.0))

BASE : The capital of Italy is Rome.
LORA : Ahoy matey! Rome be the capital of Italy. Arr, that be all!


In [174]:
from openai import OpenAI
client = OpenAI(base_url=f"http://localhost:{port}/v1", api_key="none")

resp = client.chat.completions.create(
    model="any",
    messages=[{"role": "user", "content": q}],
    extra_body={"lora": [{"id": 0, "scale": 1.0}]},
)
print(resp.choices[0].message.content)
resp = client.chat.completions.create(
    model="any",
    messages=[{"role": "user", "content": q}],
    extra_body={"lora": [{"id": 0, "scale": 0.0}]},
)
print(resp.choices[0].message.content)

Ahoy matey! Rome be the capital of Italy. Arr, that be all!
The capital of Italy is Rome.


# LoRA

In [1]:
# Task 2: genre LoRA adapter for Qwen3.5-4B

# Fine-tunes a LoRA adapter on `Qwen/Qwen3.5-4B` so that the model **writes the genre name** of a story.
# The base weights stay frozen. In vLLM one instance serves both tasks:
# `model="qwen3.5-4b"` for the chatbot (Task 1, no adapter) and `model="genre-lora"` for classification (Task 2).

# **Steps:** load data and split → build prompts → zero-shot baseline → LoRA training with a val check each epoch → test metrics → save the adapter for vLLM.

# **Hardware:** A100 or L4, bf16 throughout (no quantization, so the adapter is trained on exactly the weights vLLM serves).

# **Output folder** (`OUTPUT_DIR`):
# - `adapter/` holds the LoRA adapter (`adapter_config.json`, `adapter_model.safetensors`), ready for `vllm serve --lora-modules`.
# - `labels.json` and `prompt.json` hold the 100 genres and the exact system prompt. Serving must use the same ones.
# - `results.json` and `*.png` hold the metrics and plots.
# Qwen3.5 needs transformers 5.x. flash-linear-attention gives fast Gated DeltaNet kernels on GPU;
# without it transformers falls back to a slower pure-PyTorch path (if its install fails, just drop it from the line).
%pip install -q "transformers>=5.2" "peft>=0.17" accelerate datasets scikit-learn matplotlib flash-linear-attention
## 1. Settings
from pathlib import Path

MODEL_ID   = "Qwen/Qwen3.5-4B"
DATA_PATH  = None                       # None = Hugging Face dataset; or a local .csv / .parquet / .jsonl copy
DATASET_ID = "FareedKhan/1k_stories_100_genre"
SPLIT_PATH = None                       # optional splits.json from the chatbot project, so both tasks share one split
OUTPUT_DIR = Path("genre_lora_qwen35")
SEED       = 42

# Prompt
MAX_STORY_TOKENS = 2048                 # stories are ~1.2k-2.5k tokens; longer ones keep their beginning

# LoRA. Only modules vLLM supports for Qwen3.5. in_proj_qkv and in_proj_z must be trained together
# (vLLM fuses them). in_proj_a / in_proj_b / conv1d and the vision tower are left alone.
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",          # 8 full-attention layers
                  "in_proj_qkv", "in_proj_z", "out_proj",          # 24 Gated DeltaNet layers
                  "gate_proj", "up_proj", "down_proj"]             # MLP in all 32 layers

# Training. Effective batch = BATCH_SIZE * GRAD_ACCUM = 16. On an L4 use BATCH_SIZE 2, GRAD_ACCUM 8.
EPOCHS, LR, WARMUP_FRAC, WEIGHT_DECAY = 5, 1e-4, 0.05, 0.0
BATCH_SIZE, GRAD_ACCUM = 4, 4
EVAL_BATCH = 8
PATIENCE   = 2                          # stop after this many epochs without a better val macro-F1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
## 2. Data and split
import json, random
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if DATA_PATH:
    p = Path(DATA_PATH)
    df = {".csv": pd.read_csv, ".parquet": pd.read_parquet}.get(p.suffix, lambda f: pd.read_json(f, lines=True))(p)
else:
    from datasets import load_dataset
    df = load_dataset(DATASET_ID, split="train").to_pandas()
df = df[["id", "title", "genre", "story"]].astype({"title": str, "genre": str, "story": str})


def make_split(df, seed=42, val_frac=0.1, test_frac=0.2):
    # Same algorithm as storybot.data.splits.make_split: stratified per genre (7/1/2 for 10 stories).
    rng = np.random.default_rng(seed)
    split = {"train": [], "val": [], "test": []}
    for _, grp in df.groupby("genre"):
        ids = grp["id"].tolist()
        rng.shuffle(ids)
        n_test, n_val = max(1, round(test_frac * len(ids))), max(1, round(val_frac * len(ids)))
        split["test"] += ids[:n_test]
        split["val"] += ids[n_test:n_test + n_val]
        split["train"] += ids[n_test + n_val:]
    return split


split = json.loads(Path(SPLIT_PATH).read_text()) if SPLIT_PATH else make_split(df, SEED)
where = {str(i): s for s, ids in split.items() for i in ids}
df["split"] = df["id"].astype(str).map(where)
assert df["split"].notna().all(), "split file does not cover every story id"

LABELS = sorted(df["genre"].unique())
print(f"{len(df)} stories, {len(LABELS)} genres |", df["split"].value_counts().to_dict())
## 3. Prompt

# The system prompt lists all genres, so the base model already has a fair zero-shot baseline and the
# adapter only has to learn to pick the right one. The chat template is applied with `enable_thinking=False`,
# which is also how vLLM will be called. The target is the genre name followed by `<|im_end|>`, and loss is
# computed on those tokens only.
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.pad_token or "<|endoftext|>"
END = "<|im_end|>"

SYSTEM_PROMPT = ("You classify short stories by genre. Answer with exactly one genre from this list and nothing else.\n"
                 "Genres: " + "; ".join(LABELS))


def prompt_ids(title, story):
    story_ids = tok(story, add_special_tokens=False)["input_ids"][:MAX_STORY_TOKENS]
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Title: {title}\n\nStory:\n{tok.decode(story_ids)}"}]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    return tok(text, add_special_tokens=False)["input_ids"]


def example(row):
    p = prompt_ids(row.title, row.story)
    t = tok(row.genre + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "labels": [-100] * len(p) + t}


data = {s: [example(r) for r in df[df.split == s].itertuples()] for s in ("train", "val", "test")}
lens = [len(x["input_ids"]) for x in data["train"]]
print(f"train sequence length: mean {np.mean(lens):.0f}, max {max(lens)}")
print(tok.decode(data["train"][0]["input_ids"])[-400:])
## 4. Model and LoRA
from transformers import AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

# Load the same class vLLM serves (Qwen3_5ForConditionalGeneration). The adapter keys then come out as
# ...model.language_model.layers..., which is the layout vLLM expects. A text-only load would save
# ...model.layers... keys, and vLLM would load that adapter as a silent no-op.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to(device)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                      target_modules=TARGET_MODULES, exclude_modules=r".*visual.*", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

adapted = [n for n, m in model.named_modules() if hasattr(m, "lora_A") and n.endswith(tuple(TARGET_MODULES))]
assert adapted and all("language_model" in n for n in adapted), "LoRA must sit only on the language model"
print(len(adapted), "layers adapted, e.g.", adapted[0])
## 5. Evaluation

# Greedy generation of a few tokens, then exact match against the label list. Outputs that are not an exact
# genre are mapped to the closest one and counted as `invalid_rate`. In vLLM, `structured_outputs` restricts the
# output to the 100 labels, so invalid outputs can't happen there.
import difflib, time
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

lower = {g.lower(): g for g in LABELS}


@torch.no_grad()
def predict(examples, use_adapter=True):
    model.eval()
    tok.padding_side = "left"
    preds, invalid = [], 0
    for i in range(0, len(examples), EVAL_BATCH):
        prompts = [[t for t, l in zip(x["input_ids"], x["labels"]) if l == -100] for x in examples[i:i + EVAL_BATCH]]
        batch = tok.pad({"input_ids": prompts}, return_tensors="pt").to(device)
        gen_kwargs = dict(**batch, max_new_tokens=16, do_sample=False, use_cache=True, pad_token_id=tok.pad_token_id)
        if use_adapter:
            out = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                out = model.generate(**gen_kwargs)
        for text in tok.batch_decode(out[:, batch["input_ids"].shape[1]:], skip_special_tokens=True):
            text = text.strip()
            if text.lower() in lower:
                preds.append(lower[text.lower()])
            else:
                invalid += 1
                preds.append((difflib.get_close_matches(text, LABELS, n=1, cutoff=0) or [LABELS[0]])[0])
    tok.padding_side = "right"
    return preds, invalid / max(1, len(examples))


def evaluate(split_name, use_adapter=True):
    gold = df[df.split == split_name]["genre"].tolist()
    preds, invalid = predict(data[split_name], use_adapter)
    return {"accuracy": accuracy_score(gold, preds), "macro_f1": f1_score(gold, preds, average="macro", zero_division=0),
            "invalid_rate": invalid}, gold, preds


t0 = time.time()
baseline, *_ = evaluate("val", use_adapter=False)
print("zero-shot base model on val:", baseline, f"({time.time() - t0:.0f}s)")
## 6. Training
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup


def collate(batch):
    n = max(len(x["input_ids"]) for x in batch)
    ids = torch.full((len(batch), n), tok.pad_token_id)
    labels = torch.full((len(batch), n), -100)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, x in enumerate(batch):
        k = len(x["input_ids"])
        ids[i, :k], labels[i, :k], mask[i, :k] = torch.tensor(x["input_ids"]), torch.tensor(x["labels"]), 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}


loader = DataLoader(data["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(SEED))
steps = EPOCHS * -(-len(loader) // GRAD_ACCUM)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
sched = get_cosine_schedule_with_warmup(opt, int(WARMUP_FRAC * steps), steps)

ADAPTER_DIR = OUTPUT_DIR / "adapter"
history = [{"epoch": 0, "train_loss": None, **{f"val_{k}": v for k, v in baseline.items()}}]
best_f1, bad_epochs, t_train = -1.0, 0, time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for step, batch in enumerate(loader):
        loss = model(**{k: v.to(device) for k, v in batch.items()}).loss
        (loss / GRAD_ACCUM).backward()
        losses.append(loss.item())
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(loader):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)

    val, *_ = evaluate("val")
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), **{f"val_{k}": v for k, v in val.items()}})
    print(f"epoch {epoch}: loss {np.mean(losses):.4f} | val acc {val['accuracy']:.3f} "
          f"macro-F1 {val['macro_f1']:.3f} invalid {val['invalid_rate']:.2f} | {(time.time() - t_train) / 60:.1f} min")

    if val["macro_f1"] > best_f1:
        best_f1, bad_epochs = val["macro_f1"], 0
        model.save_pretrained(ADAPTER_DIR)              # adapter only (a few tens of MB)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stop"); break

train_minutes = (time.time() - t_train) / 60
print(f"best val macro-F1 {best_f1:.3f}, training took {train_minutes:.1f} min")
## 7. Test metrics (best adapter)
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

set_peft_model_state_dict(model, load_file(ADAPTER_DIR / "adapter_model.safetensors"))
test, gold, preds = evaluate("test")
test_base, *_ = evaluate("test", use_adapter=False)
print("test, base model :", test_base)
print("test, with LoRA  :", test)
p, r, f, _ = precision_recall_fscore_support(gold, preds, average="macro", zero_division=0)
print(f"macro precision {p:.3f} recall {r:.3f} F1 {f:.3f}")
print(classification_report(gold, preds, zero_division=0))

hist = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.epoch[1:], hist.train_loss[1:], marker="o"); ax[0].set(title="train loss", xlabel="epoch")
ax[1].plot(hist.epoch, hist.val_accuracy, marker="o", label="accuracy")
ax[1].plot(hist.epoch, hist.val_macro_f1, marker="o", label="macro-F1")
ax[1].set(title="val (epoch 0 = zero-shot base)", xlabel="epoch", ylim=(0, 1)); ax[1].legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=120); plt.show()

cm = confusion_matrix(gold, preds, labels=LABELS)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap="Blues"); ax.set(title="test confusion matrix (100 genres, sorted A-Z)", xlabel="predicted", ylabel="true")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=120); plt.show()

confusions = [(LABELS[i], LABELS[j], int(cm[i, j])) for i in range(len(LABELS)) for j in range(len(LABELS)) if i != j and cm[i, j]]
print("most confused (true -> predicted):", sorted(confusions, key=lambda c: -c[2])[:10])
## 8. Save for vLLM

# Checks that the adapter keys use the `language_model` layout vLLM expects and that `in_proj_qkv` and
# `in_proj_z` appear together, then writes the labels, prompt and results next to the adapter and zips everything.
import shutil

keys = list(load_file(ADAPTER_DIR / "adapter_model.safetensors").keys())
assert all(".language_model." in k for k in keys), keys[:3]
assert any("in_proj_qkv" in k for k in keys) == any("in_proj_z" in k for k in keys)
print(len(keys), "tensors, e.g.", keys[0])

(OUTPUT_DIR / "labels.json").write_text(json.dumps(LABELS, indent=1))
(OUTPUT_DIR / "prompt.json").write_text(json.dumps({
    "system_prompt": SYSTEM_PROMPT, "user_template": "Title: {title}\n\nStory:\n{story}",
    "max_story_tokens": MAX_STORY_TOKENS, "enable_thinking": False}, indent=1))
(OUTPUT_DIR / "results.json").write_text(json.dumps({
    "model": MODEL_ID, "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES},
    "training": {"epochs_run": len(history) - 1, "lr": LR, "effective_batch": BATCH_SIZE * GRAD_ACCUM,
                 "minutes": round(train_minutes, 1), "gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu"},
    "history": history, "test_base": test_base, "test_lora": test}, indent=1, default=float))

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("saved", archive)
## 9. Serving it (reference)

# ```bash
# vllm serve Qwen/Qwen3.5-4B --served-model-name qwen3.5-4b --language-model-only \
#   --max-model-len 32768 --reasoning-parser qwen3 \
#   --enable-auto-tool-choice --tool-call-parser qwen3_coder \
#   --default-chat-template-kwargs '{"enable_thinking": false}' \
#   --enable-lora --max-loras 1 --max-lora-rank 16 \
#   --lora-modules genre-lora=genre_lora_qwen35/adapter
# ```

# ```python
# from openai import OpenAI
# cfg, labels = json.load(open("genre_lora_qwen35/prompt.json")), json.load(open("genre_lora_qwen35/labels.json"))
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
# r = client.chat.completions.create(
#     model="genre-lora",                                    # "qwen3.5-4b" = base model for the chatbot
#     messages=[{"role": "system", "content": cfg["system_prompt"]},
#               {"role": "user", "content": cfg["user_template"].format(title=title, story=story)}],  # story cut to max_story_tokens
#     temperature=0, max_tokens=16,
#     extra_body={"structured_outputs": {"choice": labels}})
# print(r.choices[0].message.content)
# ```

1000 stories, 99 genres | {'train': 700, 'test': 200, 'val': 100}
train sequence length: mean 1568, max 2433
nolds, sailed onward into the stars, their journey far from over. For they knew that there were still countless mysteries to uncover and challenges to face in the vast expanse of the cosmos. And as long as there were heroes to answer the call, the universe would remain a place of hope, wonder, and boundless possibilities.<|im_end|>
<|im_start|>assistant
<think>

</think>

Science Fiction<|im_end|>


W1007 05:38:45.392000 118139 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1007 05:38:45.439000 118139 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

trainable params: 30,474,240 || all params: 4,569,739,776 || trainable%: 0.6669
200 layers adapted, e.g. base_model.model.model.language_model.layers.0.linear_attn.out_proj


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


zero-shot base model on val: {'accuracy': 0.39, 'macro_f1': 0.3370217324762779, 'invalid_rate': 0.01} (20s)


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.93 GiB. GPU 0 has a total capacity of 39.49 GiB of which 1013.44 MiB is free. Including non-PyTorch memory, this process has 38.49 GiB memory in use. Of the allocated memory 28.24 GiB is allocated by PyTorch, and 9.07 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)